In [11]:
import json
import numpy as np
from pathlib import Path
from common import OUTPUT_DIM_NOTES

# Define the directory containing JAMS files. The corresponding input audio files have identical names but with .wav extension. These names are deduced
# from the jams files automatically.
# jams_dirs=['/home/gerald/guitarset/annotation/rock']
jams_dirs=['/data/SynthTab/stratocaster_clean_bridge_jams']
input_audio='/data/SynthTab/stratocaster_clean_bridge'
output_data_dir='/data/data_slices/training'

def load_jams_file(file_path):
    with open(file_path, 'r') as f:
        jams_data = json.load(f)
    return jams_data

#Extract from synthtab jams file the note_tab annotations
def extract_guitar_annotations(jams_data,duration_bin_s=1):
    annotations=[]
    for annotation in jams_data['annotations']:
        if annotation['namespace']=='note_tab':
            base_tuning=annotation['sandbox']["open_tuning"]
            data=annotation['data']
            for note in data:
                annotations.append({
                    'time':note['time'],
                    'duration':note['duration'],
                    'midi_note':note['value']['fret']+base_tuning,
                    'confidence':note['confidence']
                })
    max_time=max(note['time']+note['duration'] for note in annotations)

    num_bins=int(np.ceil(max_time/duration_bin_s))
    binned_annotations=[[] for _ in range(num_bins)]
    for note in annotations:
        bin_idx=int(note['time']//duration_bin_s)
        binned_annotations[bin_idx].append(note)
    return binned_annotations,num_bins,max_time

def create_pianoroll(annotations, sr=48000, hop_length=256,  max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    max_time=max(note['time']+note['duration'] for note in annotations)
    frame_time=1/sr
    num_frames=int(np.ceil(max_time *sr))
    num_notes=max_note
    piano_roll=np.zeros((num_frames,num_notes))
    for note in annotations:
        start_frame=int(np.floor(note['time']*sr))
        end_frame=int(np.ceil((note['time']+note['duration'])*sr))
        note_idx=int(note['midi_note'])
        if note_idx>=0 and note_idx<num_notes:
            piano_roll[start_frame:end_frame,note_idx]=1
    piano_roll=np.swapaxes(piano_roll,0,1)
    onsets=np.zeros_like(piano_roll)
    onsets[:,1:]=np.diff(piano_roll,axis=1)
    onsets=np.max(onsets,axis=0)
    onsets[np.where(onsets<0)[0]]=1
    if onsets_2d:
        onsets=np.tile(onsets,(OUTPUT_DIM_NOTES,1))
    
    for t in range(piano_roll.shape[1]):
        notes=piano_roll[range(0,OUTPUT_DIM_NOTES),t].max()
        if notes==0:
            piano_roll[OUTPUT_DIM_NOTES-1,t]=1

    return piano_roll,onsets

def load_jams_annotations(file_path, sr=48000, hop_length=256, max_note=OUTPUT_DIM_NOTES,onsets_2d=False):
    jams_data=load_jams_file(file_path)
    return extract_guitar_annotations(jams_data)


In [14]:
from fretboard import FretBoard
from common import frame_size,reshape_to_nn_input,reshape_to_nn_output,save_data_slices
import os
from os import path
from scipy import io
import glob 
# Files where converted with for file in *.wav; do sox -SG "$file" -r 48000 -e float -b 32 48k/"$file"; done


def prepare_audio_midi_data(jams_file):

    # Compute the piano roll and onsets
    print(f"Processing {jams_file}")
    binned_annotations,num_bins,max_time=load_jams_annotations(jams_file)
    basename=os.path.dirname(jams_file).replace('_jams','')
    print(f"Number of time bins: {num_bins}, max time: {max_time}")
    print(f"Saving binned annotations for {basename}")
    # # We want to cut out sections around the onsets
    # # Compute the positions to preserve
    # onset_positions=np.where(onsets>0)[0]
    # for onset in onset_positions:
    #     onsets[(onset-4*frame_size):onset+4*frame_size]=1

    # preserve_indices=np.where(onsets==0)[0]

    # #Output directory
    # basename=os.path.basename(jams_file).replace('.jams','')
    # dirname=os.path.join(output_data_dir,basename)
    
    # #Input audio
    # audio_file=path.join(input_audio,basename+'_mix.wav')
    # print("Loading audio file ",audio_file)
    # sample_rate,audio=io.wavfile.read(audio_file)

    # # Normalize the audio
    # maxaudio=np.max(np.abs(audio))
    # audio=audio/maxaudio
    # plt.plot(audio)
    # plt.show()

    # # Check the sample rate
    # if sample_rate!=48000:
    #     print("Unexpected sample rate ",sample_rate)
    #     return
    
    # # Match piano roll length to audio length
    # lenpianorool=piano_roll.shape[1]
    # desired_length=len(audio)
    # if lenpianorool<desired_length:
    #     # Pad with zeros
    #     padding = desired_length - lenpianorool
    #     piano_roll = np.pad(piano_roll, ((0, 0), (0, padding)), mode='constant')
    # elif lenpianorool>desired_length:
    #     # Truncate
    #     piano_roll = piano_roll[:, :desired_length]

    # # Compute the filterbank features
    # filter =FretBoard(17.5,sample_rate)
    # numfilters=filter.get_num_filters()
    # filterbank_out=np.zeros((numfilters,len(audio)))
    # filter.process(audio,filterbank_out)

    # # Prune the onset features of the filterbank output and piano roll
    # filterbank_out=np.take(filterbank_out,preserve_indices,axis=1,mode='clip')

    # piano_roll=np.take(piano_roll,preserve_indices,axis=1,mode='clip')
    # print("Sizes after onset pruning:"+str(filterbank_out.shape)+", "+str(piano_roll.shape))
    # assert filterbank_out.shape[1]==piano_roll.shape[1],"Mismatched shapes after onset pruning"

    # # Visualize the pruned data
    # # display(plot_heatmap(piano_roll))
    # # display(plot_heatmap(filterbank_out))

    # # Reshape to NN input/output format
    # input_slices=reshape_to_nn_input(filterbank_out)
    
    
    
    # # Create output directories
    # outdir=os.path.join(dirname,'output')
    # indir=os.path.join(dirname,'input')
    
    
    # Path(outdir).mkdir(parents=True,exist_ok=True)
    # Path(indir).mkdir(parents=True,exist_ok=True)

    # # save the data slices
    # print("Saving input data to ",indir)
    # save_data_slices(indir,input_slices,1)
    
    # print("Saving output data to ",outdir)
    # output_slices=reshape_to_nn_output(piano_roll,frame_size)
    # save_data_slices(outdir,output_slices,1)

from joblib import Parallel, delayed

# Flatten files as shown above
all_jams_files = [f for d in jams_dirs for f in sorted(glob.glob(os.path.join(d,'**', '*.jams'),recursive=True))]

#Parallel(n_jobs=5)(delayed(prepare_audio_midi_data)(f) for f in all_jams_files) 
    
for jams_file in all_jams_files:
    prepare_audio_midi_data(jams_file)
# for jams_dir in jams_dirs:
#     jams_files=sorted(glob.glob(os.path.join(jams_dir, '*.jams')))

#     for jams_file in jams_files:
#         prepare_audio_midi_data(jams_file)
    


Processing /data/SynthTab/stratocaster_clean_bridge_jams/(HED)P - E -  - I Got You - gp4__3 - Electric Jazz Guitar__midi/stratocaster_clean_bridge_nonoise_stere.jams
Number of time bins: 240000, max time: 240000.0
Saving binned annotations for /data/SynthTab/stratocaster_clean_bridge/(HED)P - E -  - I Got You - gp4__3 - Electric Jazz Guitar__midi
Processing /data/SynthTab/stratocaster_clean_bridge_jams/12 Stones - Crash - gp4__1 - Electric Clean Guitar__midi/stratocaster_clean_bridge_nonoise_stere.jams
Number of time bins: 518880, max time: 518880.0
Saving binned annotations for /data/SynthTab/stratocaster_clean_bridge/12 Stones - Crash - gp4__1 - Electric Clean Guitar__midi
Processing /data/SynthTab/stratocaster_clean_bridge_jams/1974 ad, pahilo junema - Manta Mero Nepali - gp3__3 - Electric Clean Guitar__midi/stratocaster_clean_bridge_nonoise_stere.jams
Number of time bins: 139200, max time: 139200.0
Saving binned annotations for /data/SynthTab/stratocaster_clean_bridge/1974 ad, pahi

ValueError: max() iterable argument is empty